# Module 9: Policy Evaluation – Chimera vs. Popularity Baseline A/B Test

## Purpose
Compare two real recommendation policies on observable metrics without artificial purchase simulation:
- **Control**: Popularity Baseline (most frequently purchased items globally)
- **Treatment**: Chimera (utility-optimized recommendations)
- **Metric**: Average Normalized Margin of the Top-5 recommended items per household

## Key Deliverables
- Randomized A/B test with 50/50 household assignment
- Welch's t-test and bootstrapped 95% confidence intervals
- Archetype-stratified impact analysis
- Budget allocation optimization via margin delta ranking
- Power analysis showing what we can detect with current sample
- Clear caveats: this measures recommendation composition, not actual purchases

## Success Criteria
- Primary metric: avg_recommended_margin per household
- Statistical significance: p-value < 0.05 (two-tailed Welch's test)
- Effect size interpretable: Cohen's d and lift confidence interval
- No artificial simulation: outcomes derived from real recommendation rankings
- Transparent assumptions: 100% acceptance of recommendations (upper bound)


## Section 1 Load Packages

In [46]:
# Section 1: Setup - Imports, Environment, and Paths

import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Data & computation
import numpy as np
import pandas as pd
from scipy import stats

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Local imports
SRC_DIR = (Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent) / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_loader import find_repo_root

ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "02_processed"
NOTEBOOK_DIR = ROOT / "notebooks"
REPORTS_DIR = ROOT / "reports" / "figures"

# Ensure directories exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ All imports successful")
print(f"✓ Root directory: {ROOT}")
print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Reports directory: {REPORTS_DIR}")


✓ All imports successful
✓ Root directory: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey
✓ Data directory: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed
✓ Reports directory: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\reports\figures


## Section 5: Randomization and Assignment

Randomize eligible households into Control (Popularity) and Treatment (Chimera) arms 50/50, then build outcome table.


In [47]:
# Section 2: Load Data Artifacts for Policy Evaluation

# 1. Chimera Top-5 Recommendations (Treatment arm)
chimera_top5 = pd.read_csv(DATA_DIR / "top5_recommendations_module3.csv")
print(f"✓ Loaded Chimera top-5 recommendations: {chimera_top5.shape}")
print(f"  Columns: {chimera_top5.columns.tolist()}")

# 2. Household margin shift data (to get eligible households and baseline margin)
margin_shift = pd.read_csv(DATA_DIR / "module6_margin_shift_chimera.csv")
eligible_households = margin_shift["household_key"].unique()
print(f"\n✓ Loaded eligible households: {len(eligible_households)}")

# 3. Commodity margin lookup (for normalized margin)
commodity_margin = pd.read_csv(DATA_DIR / "commodity_margin.csv")
print(f"\n✓ Loaded commodity margins: {commodity_margin.shape}")
print(f"  Key columns: {commodity_margin.columns.tolist()}")

# 4. Master transactions (for building Popularity baseline)
# Load full transactions but then filter to training window to avoid look-ahead bias
master_txn = pd.read_parquet(DATA_DIR / "master_transactions.parquet")
print(f"\n✓ Loaded master transactions: {master_txn.shape}")
print(f"  Day range: {master_txn['DAY'].min()} - {master_txn['DAY'].max()}")

# IMPORTANT: restrict to training window to avoid look-ahead bias
TRAIN_END_DAY = 600
train_txn = master_txn[master_txn["DAY"] <= TRAIN_END_DAY].copy()
print(f"\n✓ Training transactions filtered to DAY <= {TRAIN_END_DAY}: {train_txn.shape}")

# 5. Archetype assignments (for balance checks and stratified analysis)
archetype_df = pd.read_csv(DATA_DIR / "module8_archetype_assignments.csv")
print(f"\n✓ Loaded archetype assignments: {archetype_df.shape}")

print("\n" + "="*80)
print("Sample of key data:")
print("\nChimera top-5:")
print(chimera_top5.head(2))
print("\nCommodity margin:")
print(commodity_margin.head(2))


✓ Loaded Chimera top-5 recommendations: (12500, 22)
  Columns: ['household_key', 'COMMODITY_DESC', 'seed_items', 'relevance_als', 'relevance_mba', 'Relevance', 'source', 'source_detail', 'snapshot_week', 'was_recently_purchased', 'habit_strength', 'Uplift', 'Normalized_Margin', 'deal_sensitivity', 'is_active_campaign_period', 'item_is_promoted', 'Context', 'Utility', 'Relevance_Contribution', 'Uplift_Contribution', 'Margin_Contribution', 'Context_Contribution']

✓ Loaded eligible households: 2408

✓ Loaded commodity margins: (308, 3)
  Key columns: ['COMMODITY_DESC', 'Raw_Margin', 'Normalized_Margin']

✓ Loaded master transactions: (2216946, 31)
  Day range: 1 - 711

✓ Training transactions filtered to DAY <= 600: (1824975, 31)

✓ Loaded archetype assignments: (2408, 6)

Sample of key data:

Chimera top-5:
   household_key           COMMODITY_DESC  \
0              1                  POPCORN   
1              1  VEGETABLES - ALL OTHERS   

                                          seed

## Section 6: Hypothesis Testing – Welch's t-test and Bootstrapped CI

Compare treatment vs control arms using classical statistical testing without assumptions of equal variance.


In [48]:
def build_popularity_baseline(train_txn, margin_lookup, eligible_hh, recent_period_days=28):
    """
    Build popularity baseline: top-5 global commodities (excluding recently purchased) per household.
    
    Args:
        train_txn: Training period transactions
        margin_lookup: Commodity margin lookup table
        eligible_hh: Array of eligible household IDs
        recent_period_days: Recency exclusion window (days)
    
    Returns:
        DataFrame with columns: household_key, COMMODITY_DESC, rank, Normalized_Margin
    """
    # Global frequency of commodities (FROM TRAINING WINDOW)
    global_freq = train_txn.groupby("COMMODITY_DESC").size().sort_values(ascending=False)
    
    # Recent purchases per household (items to exclude) - use train_txn's max day
    max_day = train_txn["DAY"].max()
    recent_cutoff = max_day - recent_period_days
    recent = train_txn[train_txn["DAY"] > recent_cutoff]
    recent_items = recent.groupby("household_key")["COMMODITY_DESC"].apply(set).to_dict()
    
    # Build recommendations
    rows = []
    for hh in eligible_hh:
        excluded = recent_items.get(hh, set())
        # Top 5 frequent items not in exclusion set
        candidates = global_freq[~global_freq.index.isin(excluded)].head(5)
        for rank, (commodity, freq) in enumerate(candidates.items(), start=1):
            rows.append({
                "household_key": hh,
                "COMMODITY_DESC": commodity,
                "rank": rank
            })
    
    df = pd.DataFrame(rows)
    
    # Merge with margin lookup
    df = df.merge(
        margin_lookup[["COMMODITY_DESC", "Normalized_Margin"]],
        on="COMMODITY_DESC",
        how="left"
    )
    
    # Fill missing and clip to [0, 1]
    df["Normalized_Margin"] = df["Normalized_Margin"].fillna(0.0).clip(0, 1)
    
    return df

# Build popularity baseline using TRAINING transactions (fixes look-ahead bias)
print("Building Popularity baseline from training window (no look-ahead)...")
popularity_baseline = build_popularity_baseline(
    train_txn,
    commodity_margin,
    eligible_households,
    recent_period_days=28
)

print(f"\n✓ Popularity baseline built: {popularity_baseline.shape}")
print(f"  Households covered: {popularity_baseline['household_key'].nunique()}")
print(f"  Sample:")
print(popularity_baseline.head(5))


Building Popularity baseline from training window (no look-ahead)...

✓ Popularity baseline built: (12040, 4)
  Households covered: 2408
  Sample:
   household_key          COMMODITY_DESC  rank  Normalized_Margin
0              1             SOFT DRINKS     1                1.0
1              1                    BEEF     2                1.0
2              1            FROZEN PIZZA     3                1.0
3              1       COUPON/MISC ITEMS     4                1.0
4              1  FRZN MEAT/MEAT DINNERS     5                1.0


In [49]:
# Section 4: Compute Primary Metric – Average Recommended Margin per Household

# Treatment Arm (Chimera)
treatment_margin = (
    chimera_top5
    .groupby("household_key")["Normalized_Margin"]
    .mean()
    .reset_index(name="avg_recommended_margin")
)

print("="*80)
print("TREATMENT ARM (Chimera)")
print("="*80)
print(f"✓ Households with Chimera recommendations: {treatment_margin.shape[0]}")
print(f"  Mean recommended margin: {treatment_margin['avg_recommended_margin'].mean():.4f}")
print(f"  Std dev: {treatment_margin['avg_recommended_margin'].std():.4f}")
print(f"  Min/Max: {treatment_margin['avg_recommended_margin'].min():.4f} / {treatment_margin['avg_recommended_margin'].max():.4f}")

# Control Arm (Popularity Baseline)
control_margin = (
    popularity_baseline
    .groupby("household_key")["Normalized_Margin"]
    .mean()
    .reset_index(name="avg_recommended_margin")
)

print("\n" + "="*80)
print("CONTROL ARM (Popularity Baseline)")
print("="*80)
print(f"✓ Households with Popularity recommendations: {control_margin.shape[0]}")
print(f"  Mean recommended margin: {control_margin['avg_recommended_margin'].mean():.4f}")
print(f"  Std dev: {control_margin['avg_recommended_margin'].std():.4f}")
print(f"  Min/Max: {control_margin['avg_recommended_margin'].min():.4f} / {control_margin['avg_recommended_margin'].max():.4f}")

print("\nSample metrics:")
print("\nTreatment sample:")
print(treatment_margin.head(3))
print("\nControl sample:")
print(control_margin.head(3))


TREATMENT ARM (Chimera)
✓ Households with Chimera recommendations: 2500
  Mean recommended margin: 0.9871
  Std dev: 0.0575
  Min/Max: 0.2000 / 1.0000

CONTROL ARM (Popularity Baseline)
✓ Households with Popularity recommendations: 2408
  Mean recommended margin: 0.9437
  Std dev: 0.0900
  Min/Max: 0.8000 / 1.0000

Sample metrics:

Treatment sample:
   household_key  avg_recommended_margin
0              1                     1.0
1              2                     1.0
2              3                     1.0

Control sample:
   household_key  avg_recommended_margin
0              1                     1.0
1              2                     0.8
2              3                     1.0


## Section 4: Power Analysis and Sample Size Calculation

Compute required sample size to detect a meaningful lift in incremental margin with 80% power.

Note: the historical sample is underpowered for small effects, so the simulated A/B test below is a proof-of-concept and should not be read as definitive causal evidence.

In [50]:
# Section 5: Randomized Assignment

# Get all eligible households
all_hh = np.array(eligible_households)
np.random.seed(42)
np.random.shuffle(all_hh)

# 50/50 split
split_idx = len(all_hh) // 2
treatment_ids = set(all_hh[:split_idx])
control_ids = set(all_hh[split_idx:])

print("="*80)
print("RANDOMIZATION RESULTS")
print("="*80)
print(f"Total eligible households: {len(all_hh)}")
print(f"Treatment group (Chimera): {len(treatment_ids)}")
print(f"Control group (Popularity): {len(control_ids)}")

# Build outcome table
treat_map = treatment_margin.set_index("household_key")["avg_recommended_margin"]
ctrl_map = control_margin.set_index("household_key")["avg_recommended_margin"]

outcomes = pd.DataFrame({"household_key": all_hh})
outcomes["treatment_arm"] = outcomes["household_key"].apply(
    lambda x: "Treatment" if x in treatment_ids else "Control"
)
outcomes["recommended_margin"] = outcomes["household_key"].apply(
    lambda x: treat_map.get(x, np.nan) if x in treatment_ids else ctrl_map.get(x, np.nan)
)

# Remove households with missing data
outcomes_clean = outcomes.dropna(subset=["recommended_margin"])

print(f"\n✓ Outcomes table: {outcomes_clean.shape[0]} households with valid metrics")
print(f"  Control: {len(outcomes_clean[outcomes_clean['treatment_arm'] == 'Control'])}")
print(f"  Treatment: {len(outcomes_clean[outcomes_clean['treatment_arm'] == 'Treatment'])}")

# Balance check: merge with archetype
outcomes_clean = outcomes_clean.merge(
    archetype_df[["household_key", "archetype"]],
    on="household_key",
    how="left"
)

print("\n" + "="*80)
print("BALANCE CHECK: Distribution by Archetype")
print("="*80)
balance_table = pd.crosstab(
    outcomes_clean["archetype"],
    outcomes_clean["treatment_arm"],
    margins=True
)
print(balance_table)
print("\n✓ Arms are well-balanced across archetypes")

print("\nSample outcomes:")
print(outcomes_clean.head(5))


RANDOMIZATION RESULTS
Total eligible households: 2408
Treatment group (Chimera): 1204
Control group (Popularity): 1204

✓ Outcomes table: 2408 households with valid metrics
  Control: 1204
  Treatment: 1204

BALANCE CHECK: Distribution by Archetype
treatment_arm         Control  Treatment   All
archetype                                     
Deal-Driven Explorer      423        412   835
Frugal Loyalist           182        187   369
Premium Discoverer        178        191   369
Routine Replenisher       421        414   835
All                      1204       1204  2408

✓ Arms are well-balanced across archetypes

Sample outcomes:
   household_key treatment_arm  recommended_margin             archetype
0           2266     Treatment                 0.8  Deal-Driven Explorer
1           2251     Treatment                 1.0  Deal-Driven Explorer
2           1435     Treatment                 1.0    Premium Discoverer
3           1593     Treatment                 1.0   Routine Repleni

## Section 5: Randomized Assignment and Outcome Simulation

Randomly assign households to control (Popularity baseline) and treatment (Chimera), then simulate outcomes.

In [51]:
# Step 9.1: Prepare cohort and Randomized 50:50 household assignment

# Build cohort (if not already present in kernel)
if 'cohort' not in globals():
    cohort = margin_shift[[
        "household_key",
        "train_avg_margin",
        "test_avg_margin",
        "train_items",
        "test_items",
        "train_baskets",
        "test_baskets"
    ]].copy()
    cohort = cohort.merge(
        archetype_df[["household_key", "archetype", "deal_sensitivity", "basket_diversity"]],
        on="household_key",
        how="left"
    )
    rec_count = chimera_top5.groupby("household_key").size().reset_index(name="n_recommendations")
    cohort = cohort.merge(rec_count, on="household_key", how="left")
    cohort = cohort[cohort["test_baskets"] > 0].copy()
    cohort["archetype_label"] = cohort["archetype"].fillna("Unknown")
    cohort["deal_sensitivity"] = pd.to_numeric(cohort["deal_sensitivity"], errors="coerce").fillna(cohort["deal_sensitivity"].median())
    cohort["basket_diversity"] = pd.to_numeric(cohort["basket_diversity"], errors="coerce").fillna(cohort["basket_diversity"].median())
    cohort["n_recommendations"] = pd.to_numeric(cohort["n_recommendations"], errors="coerce").fillna(0)
    cohort["incremental_margin_observed"] = cohort["test_avg_margin"] - cohort["train_avg_margin"]
    print(f"✓ Cohort prepared: {cohort.shape[0]} households")

# Create assignment dataframe
assignment = cohort[[
    "household_key",
    "train_avg_margin",
    "test_avg_margin",
    "incremental_margin_observed",
    "archetype_label",
    "deal_sensitivity",
    "basket_size_change" if 'basket_size_change' in cohort.columns else 'train_baskets',
    "train_items",
    "test_items",
    "train_baskets",
    "test_baskets",
    "basket_diversity",
    "n_recommendations"
]].copy()

# Perform 50:50 randomization
try:
    control_ids, treatment_ids = random_split_households(
        assignment["household_key"],
        control_fraction=0.5,
        random_seed=42
    )
except Exception:
    # Fallback: simple numpy split if helper not available
    all_hh = assignment["household_key"].to_numpy().copy()
    np.random.seed(42)
    np.random.shuffle(all_hh)
    split_idx = len(all_hh)//2
    treatment_ids = set(all_hh[:split_idx])
    control_ids = set(all_hh[split_idx:])

assignment["treatment_arm"] = assignment["household_key"].apply(
    lambda x: "Control" if x in control_ids else "Treatment"
)

print("Randomization Results:")
print("="*80)
print(f"Total households: {len(assignment)}")
print(f"Control group (Popularity): {len(control_ids)}")
print(f"Treatment group (Chimera): {len(treatment_ids)}")

# Check balance by archetype
print("\nBalance check by archetype:")
balance_check = pd.crosstab(
    assignment["archetype_label"],
    assignment["treatment_arm"],
    margins=True
)
print(balance_check)

# Save assignment mapping: prefer outcomes_clean if present to avoid duplicate/misaligned columns
if 'outcomes_clean' in globals():
    outcomes_clean.to_csv(DATA_DIR / "ab_assignment_mapping.csv", index=False)
    print(f"\n✓ ab_assignment_mapping.csv saved from outcomes_clean (clean export)")
else:
    assignment.to_csv(DATA_DIR / "ab_assignment_mapping.csv", index=False)
    print(f"\n✓ ab_assignment_mapping.csv saved from assignment (fallback)")
    
assignment.to_csv(DATA_DIR / "ab_assignment_full_mapping.csv", index=False)
print(f"\n✓ ab_assignment_full_mapping.csv saved from assignment (fallback)")    


Randomization Results:
Total households: 2408
Control group (Popularity): 1204
Treatment group (Chimera): 1204

Balance check by archetype:
treatment_arm         Control  Treatment   All
archetype_label                               
Deal-Driven Explorer      423        412   835
Frugal Loyalist           182        187   369
Premium Discoverer        178        191   369
Routine Replenisher       421        414   835
All                      1204       1204  2408

✓ ab_assignment_mapping.csv saved from outcomes_clean (clean export)

✓ ab_assignment_full_mapping.csv saved from assignment (fallback)


In [52]:
# Preprocessing: Build analysis cohort required for assignment
# Create a cohort dataframe from margin_shift, archetype assignments and recommendation counts
cohort = margin_shift[[
    "household_key",
    "train_avg_margin",
    "test_avg_margin",
    "train_items",
    "test_items",
    "train_baskets",
    "test_baskets"
]].copy()

# Merge archetype and other features
cohort = cohort.merge(
    archetype_df[["household_key", "archetype", "deal_sensitivity", "basket_diversity"]],
    on="household_key",
    how="left"
)

# Recommendation counts as a proxy feature
rec_count = chimera_top5.groupby("household_key").size().reset_index(name="n_recommendations")
cohort = cohort.merge(rec_count, on="household_key", how="left")

# Filter households with test baskets > 0
cohort = cohort[cohort["test_baskets"] > 0].copy()

# Clean and fill defaults
cohort["archetype_label"] = cohort["archetype"].fillna("Unknown")
cohort["deal_sensitivity"] = pd.to_numeric(cohort["deal_sensitivity"], errors="coerce").fillna(cohort["deal_sensitivity"].median())
cohort["basket_diversity"] = pd.to_numeric(cohort["basket_diversity"], errors="coerce").fillna(cohort["basket_diversity"].median())
cohort["n_recommendations"] = pd.to_numeric(cohort["n_recommendations"], errors="coerce").fillna(0)

# Compute observed incremental margin
cohort["incremental_margin_observed"] = cohort["test_avg_margin"] - cohort["train_avg_margin"]

print(f"✓ Cohort prepared: {cohort.shape[0]} households")
print(cohort.head(3))


✓ Cohort prepared: 2408 households
   household_key  train_avg_margin  test_avg_margin  train_items  test_items  \
0              1          0.945174         0.942130        121.0        94.0   
1              2          0.890566         0.896739        130.0        78.0   
2              3          0.957895         0.925373        108.0        24.0   

   train_baskets  test_baskets             archetype  deal_sensitivity  \
0           65.0          21.0    Premium Discoverer          0.823529   
1           37.0           8.0  Deal-Driven Explorer          0.933333   
2           42.0           5.0  Deal-Driven Explorer          0.978723   

   basket_diversity  n_recommendations       archetype_label  \
0         15.952381                  5    Premium Discoverer   
1         17.250000                  5  Deal-Driven Explorer   
2          7.800000                  5  Deal-Driven Explorer   

   incremental_margin_observed  
0                    -0.003044  
1                     0.

## Section 6: Statistical Testing - Welch's t-test and Bootstrapped Confidence Intervals

Test the hypothesis that treatment (Chimera) produces higher incremental margin than control (Popularity).

In [53]:
# Section 6: Hypothesis Testing – Welch's t-test and Bootstrapped CI

# Separate by arm
control_vals = outcomes_clean.loc[outcomes_clean["treatment_arm"] == "Control", "recommended_margin"].values
treatment_vals = outcomes_clean.loc[outcomes_clean["treatment_arm"] == "Treatment", "recommended_margin"].values

print("="*80)
print("STATISTICAL TESTING (Welch's t-test, Two-Sided)")
print("="*80)

# Descriptive statistics
print("\nControl Group (Popularity Baseline):")
print(f"  n = {len(control_vals)}")
print(f"  Mean = {control_vals.mean():.6f}")
print(f"  Std = {control_vals.std():.6f}")
print(f"  [Min, Max] = [{control_vals.min():.6f}, {control_vals.max():.6f}]")

print("\nTreatment Group (Chimera):")
print(f"  n = {len(treatment_vals)}")
print(f"  Mean = {treatment_vals.mean():.6f}")
print(f"  Std = {treatment_vals.std():.6f}")
print(f"  [Min, Max] = [{treatment_vals.min():.6f}, {treatment_vals.max():.6f}]")

# Point estimate of lift
lift = treatment_vals.mean() - control_vals.mean()
print(f"\nPoint Estimate of Lift (Treatment - Control):")
print(f"  Δ = {lift:.6f}")

# Welch's t-test (two-sided, unequal variance)
t_stat, p_value = stats.ttest_ind(treatment_vals, control_vals, equal_var=False)
print(f"\nWelch's t-test (H₀: μ_treatment = μ_control):")
print(f"  t-statistic = {t_stat:.6f}")
print(f"  p-value (two-sided) = {p_value:.6f}")
print(f"  Significant at α=0.05? {'✓ YES' if p_value < 0.05 else '✗ NO'}")

# Cohen's d effect size
pooled_std = np.sqrt(
    ((len(treatment_vals) - 1) * treatment_vals.std()**2 +
     (len(control_vals) - 1) * control_vals.std()**2) /
    (len(treatment_vals) + len(control_vals) - 2)
)
cohens_d = lift / pooled_std if pooled_std > 0 else 0
print(f"\nEffect Size (Cohen's d):")
print(f"  d = {cohens_d:.4f}")
effect_interpretation = (
    "negligible" if abs(cohens_d) < 0.2 else
    "small" if abs(cohens_d) < 0.5 else
    "medium" if abs(cohens_d) < 0.8 else
    "large"
)
print(f"  Interpretation: {effect_interpretation}")

# Bootstrapped 95% CI
def bootstrap_ci(treatment, control, n_boot=10_000, alpha=0.05):
    """Compute bootstrapped CI for the difference in means."""
    np.random.seed(42)
    boot_diffs = []
    for _ in range(n_boot):
        treat_boot = np.random.choice(treatment, size=len(treatment), replace=True).mean()
        ctrl_boot = np.random.choice(control, size=len(control), replace=True).mean()
        boot_diffs.append(treat_boot - ctrl_boot)
    boot_diffs = np.array(boot_diffs)
    ci_lower = np.percentile(boot_diffs, 100*alpha/2)
    ci_upper = np.percentile(boot_diffs, 100*(1-alpha/2))
    return ci_lower, ci_upper

ci_lower, ci_upper = bootstrap_ci(treatment_vals, control_vals)
print(f"\nBootstrapped 95% Confidence Interval (10,000 iterations):")
print(f"  CI = [{ci_lower:.6f}, {ci_upper:.6f}]")
print(f"  Includes zero? {'✓ YES (not significant)' if ci_lower <= 0 <= ci_upper else '✗ NO (significant)'}")

# Summary
test_results = {
    "lift": lift,
    "ci_lower": ci_lower,
    "ci_upper": ci_upper,
    "t_stat": t_stat,
    "p_value": p_value,
    "cohens_d": cohens_d,
    "control_n": len(control_vals),
    "treatment_n": len(treatment_vals),
    "control_mean": control_vals.mean(),
    "treatment_mean": treatment_vals.mean(),
    "is_significant": p_value < 0.05
}

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
if test_results["is_significant"]:
    print(f"✓ SIGNIFICANT: Chimera's recommended margin is significantly different from Popularity")
    print(f"  Direction: {'higher' if lift > 0 else 'lower'}")
    print(f"  Magnitude: {abs(lift):.6f} points ({abs(cohens_d):.3f} standard deviations)")
else:
    print(f"✗ NOT SIGNIFICANT: Cannot detect a difference between Chimera and Popularity")
    print(f"  Point estimate: {lift:.6f} points")
    print(f"  Could be due to underpowered sample or true null effect")


STATISTICAL TESTING (Welch's t-test, Two-Sided)

Control Group (Popularity Baseline):
  n = 1204
  Mean = 0.941362
  Std = 0.091045
  [Min, Max] = [0.800000, 1.000000]

Treatment Group (Chimera):
  n = 1204
  Mean = 0.988870
  Std = 0.052598
  [Min, Max] = [0.600000, 1.000000]

Point Estimate of Lift (Treatment - Control):
  Δ = 0.047508

Welch's t-test (H₀: μ_treatment = μ_control):
  t-statistic = 15.671471
  p-value (two-sided) = 0.000000
  Significant at α=0.05? ✓ YES

Effect Size (Cohen's d):
  d = 0.6390
  Interpretation: medium

Bootstrapped 95% Confidence Interval (10,000 iterations):
  CI = [0.041528, 0.053488]
  Includes zero? ✗ NO (significant)

CONCLUSION
✓ SIGNIFICANT: Chimera's recommended margin is significantly different from Popularity
  Direction: higher
  Magnitude: 0.047508 points (0.639 standard deviations)


## Section 7: Power Analysis – What Can We Detect?

Retrospective power analysis: given our sample size and observed variance, what effect size is detectable at 80% power?


In [54]:
# Section 7: Retrospective Power Analysis

def compute_detectable_effect_size(n_per_group, std_pooled, alpha=0.05, power=0.80):
    """
    Compute minimum detectable effect (MDE) given sample size, variance, alpha, and power.
    Using Welch's t-test formula.
    """
    z_alpha_half = stats.norm.ppf(1 - alpha / 2)  # Two-sided
    z_beta = stats.norm.ppf(power)
    mde = (z_alpha_half + z_beta) * std_pooled * np.sqrt(2 / n_per_group)
    return mde

# Compute with observed sample size and variance
n_per_group = min(len(control_vals), len(treatment_vals))
std_pooled = pooled_std

mde = compute_detectable_effect_size(n_per_group, std_pooled, alpha=0.05, power=0.80)

print("="*80)
print("RETROSPECTIVE POWER ANALYSIS (α=0.05, Power=0.80, Welch's t-test)")
print("="*80)

print(f"\nObserved Sample Size:")
print(f"  Per group (n): {n_per_group}")
print(f"  Total sample: {n_per_group * 2}")

print(f"\nObserved Variance:")
print(f"  Pooled std: {std_pooled:.6f}")

print(f"\nMinimum Detectable Effect (MDE) at 80% Power:")
print(f"  MDE (normalized margin index): {mde:.6f}")
print(f"  Interpretation: With {n_per_group} households per group, we can reliably")
print(f"    detect a difference ≥ {mde:.6f} in normalized margin index")

# Sensitivity analysis: what power do we achieve at different effect sizes?
effect_sizes = [0.01, 0.02, 0.05, 0.10, test_results["lift"]]
print(f"\nSensitivity Analysis: Power at different effect sizes")
print(f"-" * 80)

power_results = []
for es in effect_sizes:
    # Cohen's d
    cohens_d_es = es / std_pooled
    # Noncentrality parameter
    ncp = cohens_d_es * np.sqrt(n_per_group / 2)
    # Power (for two-sided Welch's t-test)
    crit_t = stats.t.ppf(0.975, df=2*n_per_group - 2)  # Two-sided critical value
    power_achieved = 1 - stats.nct.cdf(crit_t, df=2*n_per_group - 2, nc=ncp)
    power_results.append({
        "Effect Size (Δ)": f"{es:.6f}",
        "Cohen's d": f"{cohens_d_es:.4f}",
        "Power (%)": f"{power_achieved*100:.1f}%"
    })

power_table = pd.DataFrame(power_results)
print(power_table.to_string(index=False))

print(f"\n" + "="*80)
print("KEY TAKEAWAY")
print("="*80)
print(f"✓ Our sample size of {n_per_group} per group can detect effects of {mde:.6f} or larger")
print(f"✓ Observed effect: {test_results['lift']:.6f}")
if abs(test_results['lift']) >= mde:
    print(f"  → DETECTABLE at 80% power (effect is large enough)")
else:
    print(f"  → BELOW minimum detectable threshold (underpowered)")
    needed_n = (stats.norm.ppf(0.975) + stats.norm.ppf(0.80))**2 * 2 * std_pooled**2 / (test_results['lift']**2)
    print(f"  → Would need ~{int(needed_n)} households per group to detect observed effect at 80% power")


RETROSPECTIVE POWER ANALYSIS (α=0.05, Power=0.80, Welch's t-test)

Observed Sample Size:
  Per group (n): 1204
  Total sample: 2408

Observed Variance:
  Pooled std: 0.074349

Minimum Detectable Effect (MDE) at 80% Power:
  MDE (normalized margin index): 0.008490
  Interpretation: With 1204 households per group, we can reliably
    detect a difference ≥ 0.008490 in normalized margin index

Sensitivity Analysis: Power at different effect sizes
--------------------------------------------------------------------------------
Effect Size (Δ) Cohen's d Power (%)
       0.010000    0.1345     91.0%
       0.020000    0.2690    100.0%
       0.050000    0.6725    100.0%
       0.100000    1.3450    100.0%
       0.047508    0.6390    100.0%

KEY TAKEAWAY
✓ Our sample size of 1204 per group can detect effects of 0.008490 or larger
✓ Observed effect: 0.047508
  → DETECTABLE at 80% power (effect is large enough)


## Section 8: Budget Allocation – Targeting by Margin Delta

Rank households by how much extra margin they gain if switched from Popularity (control) to Chimera (treatment).


In [55]:
# Section 8: Budget Allocation – Targeting by Margin Delta

# Build household-level margin data
targeting_df = (
    treatment_margin
    .merge(
        control_margin,
        on="household_key",
        how="inner",
        suffixes=("_chimera", "_popularity")
    )
)

# Compute margin delta (incremental gain from Chimera)
targeting_df["margin_delta"] = (
    targeting_df["avg_recommended_margin_chimera"] -
    targeting_df["avg_recommended_margin_popularity"]
)

# Sort by margin delta
targeting_df_sorted = targeting_df.sort_values("margin_delta", ascending=False)

print("="*80)
print("BUDGET ALLOCATION – TARGETING BY MARGIN DELTA")
print("="*80)

print(f"\nHousehold-level margin comparison:")
print(f"  Total households: {len(targeting_df)}")
print(f"  Mean margin delta: {targeting_df['margin_delta'].mean():.6f}")
print(f"  Std margin delta: {targeting_df['margin_delta'].std():.6f}")
print(f"  [Min, Max] delta: [{targeting_df['margin_delta'].min():.6f}, {targeting_df['margin_delta'].max():.6f}]")

# Top 20% targeting (assumption-free heuristic)
n_target = max(1, int(len(targeting_df) * 0.20))
top_20_pct = targeting_df_sorted.head(n_target).copy()
random_20_pct = targeting_df.sample(n=n_target, random_state=42).copy()

gain_targeted = top_20_pct["margin_delta"].sum()
gain_random = random_20_pct["margin_delta"].sum()
gain_all = targeting_df["margin_delta"].sum()

print(f"\n" + "="*80)
print("Targeting Strategy Comparison (Top 20% of households):")
print(f"-" * 80)

strategy_comp = pd.DataFrame({
    "Strategy": ["Random 20%", "Targeted Top 20% (Optimal)"],
    "Households": [len(random_20_pct), len(top_20_pct)],
    "Total Margin Delta": [f"{gain_random:.6f}", f"{gain_targeted:.6f}"],
    "Avg Per Household": [f"{gain_random/len(random_20_pct):.6f}", f"{gain_targeted/len(top_20_pct):.6f}"],
})

if gain_random > 0:
    efficiency = gain_targeted / gain_random
    strategy_comp["Efficiency vs Random"] = ["1.0x", f"{efficiency:.2f}x"]

print(strategy_comp.to_string(index=False))

print(f"\n✓ Key Finding:")
print(f"  By targeting the top 20% of households (by margin delta),")
print(f"  we capture {gain_targeted:.6f} total margin points")
print(f"  vs. {gain_random:.6f} from random targeting")
if gain_random > 0:
    print(f"  Efficiency gain: {efficiency:.2f}x")

print(f"\nTop 10 Households to Target:")
top_10 = targeting_df_sorted.head(10)[["household_key", "avg_recommended_margin_chimera", 
                                        "avg_recommended_margin_popularity", "margin_delta"]]
print(top_10.to_string(index=False))

# Save targeting list
targeting_output = top_20_pct[[
    "household_key",
    "avg_recommended_margin_chimera",
    "avg_recommended_margin_popularity",
    "margin_delta"
]].copy()
targeting_output.columns = [
    "household_key",
    "chimera_recommended_margin",
    "popularity_recommended_margin",
    "incremental_margin_delta"
]

# Round incremental margins for CSV clarity
if "incremental_margin_delta" in targeting_output.columns:
    targeting_output["incremental_margin_delta"] = targeting_output["incremental_margin_delta"].round(6)

# Save CSV
targeting_output.to_csv(DATA_DIR / "module9_optimal_targeting_top20pct.csv", index=False)
print(f"\n✓ Top 20% targeting list saved to: module9_optimal_targeting_top20pct.csv")


BUDGET ALLOCATION – TARGETING BY MARGIN DELTA

Household-level margin comparison:
  Total households: 2408
  Mean margin delta: 0.043106
  Std margin delta: 0.107015
  [Min, Max] delta: [-0.600000, 0.200000]

Targeting Strategy Comparison (Top 20% of households):
--------------------------------------------------------------------------------
                  Strategy  Households Total Margin Delta Avg Per Household Efficiency vs Random
                Random 20%         481          19.800000          0.041164                 1.0x
Targeted Top 20% (Optimal)         481          96.200000          0.200000                4.86x

✓ Key Finding:
  By targeting the top 20% of households (by margin delta),
  we capture 96.200000 total margin points
  vs. 19.800000 from random targeting
  Efficiency gain: 4.86x

Top 10 Households to Target:
 household_key  avg_recommended_margin_chimera  avg_recommended_margin_popularity  margin_delta
          2500                             1.0          

## Section 9: Visualizations – Treatment Effect and Targeting Efficiency

Create interactive charts showing confidence intervals, cumulative gain curves, and stratified impacts.


In [56]:
# Section 9: Visualizations

# ===== Figure 1: Lift with 95% Confidence Interval =====
fig_lift = go.Figure()

fig_lift.add_trace(go.Bar(
    x=["Incremental Lift\n(Chimera - Popularity)"],
    y=[test_results["lift"]],
    marker_color="#2ca02c",
    error_y=dict(
        type='data',
        symmetric=False,
        array=[test_results["ci_upper"] - test_results["lift"]],
        arrayminus=[test_results["lift"] - test_results["ci_lower"]]
    ),
    text=[f"Δ = {test_results['lift']:.6f}<br>95% CI: [{test_results['ci_lower']:.6f}, {test_results['ci_upper']:.6f}]"],
    textposition="outside",
    hovertemplate="<b>Lift</b><br>Point estimate: %{y:.6f}<br>95% CI: [%{error_y.arrayminus:.6f}, %{error_y.array:.6f}]<extra></extra>"
))

fig_lift.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="No effect", annotation_position="right")

fig_lift.update_layout(
    title=f"Treatment Effect: Chimera vs. Popularity Baseline<br><sub>Normalized Margin Index (0-1 scale)</sub>",
    yaxis_title="Average Recommended Margin",
    xaxis_title="",
    template="plotly_white",
    height=500,
    width=700,
    showlegend=False
)

fig_lift.write_html(REPORTS_DIR / "policy_eval_lift_bar.html")
print("✓ Saved: policy_eval_lift_bar.html")

# ===== Figure 2: Cumulative Margin Gain (Targeting Efficiency) =====
fig_gain = go.Figure()

# Random strategy
random_sorted = targeting_df.sample(frac=1, random_state=42).reset_index(drop=True)
random_sorted["cumulative"] = random_sorted["margin_delta"].cumsum()
random_sorted["pct_targeted"] = (np.arange(len(random_sorted)) + 1) / len(random_sorted) * 100

# Optimal strategy
optimal_sorted = targeting_df_sorted.reset_index(drop=True)
optimal_sorted["cumulative"] = optimal_sorted["margin_delta"].cumsum()
optimal_sorted["pct_targeted"] = (np.arange(len(optimal_sorted)) + 1) / len(optimal_sorted) * 100

fig_gain.add_trace(go.Scatter(
    x=random_sorted["pct_targeted"],
    y=random_sorted["cumulative"],
    mode='lines',
    name='Random',
    line=dict(color='#808080', width=2, dash='dash')
))

fig_gain.add_trace(go.Scatter(
    x=optimal_sorted["pct_targeted"],
    y=optimal_sorted["cumulative"],
    mode='lines',
    name='Optimal (By Margin Delta)',
    line=dict(color='#2ca02c', width=3)
))

# Mark 20% targeting point
fig_gain.add_vline(
    x=20,
    line_dash="dot",
    line_color="red",
    annotation_text="20% Budget",
    annotation_position="top right"
)

fig_gain.update_layout(
    title="Cumulative Margin Gain: Targeting Strategy Efficiency",
    xaxis_title="Percentage of Households Targeted (%)",
    yaxis_title="Cumulative Margin Delta",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    width=900
)

fig_gain.write_html(REPORTS_DIR / "policy_eval_cumulative_gain.html")
print("✓ Saved: policy_eval_cumulative_gain.html")

# ===== Figure 3: Stratified Impact by Archetype =====
# Merge outcomes with archetype
arch_impact = outcomes_clean.copy()

arch_summary = arch_impact.groupby(["archetype", "treatment_arm"])["recommended_margin"].agg([
    "mean", "std", "count"
]).reset_index()

fig_arch = go.Figure()

for arch in arch_summary["archetype"].unique():
    arch_data = arch_summary[arch_summary["archetype"] == arch]
    for arm in ["Control", "Treatment"]:
        arm_data = arch_data[arch_data["treatment_arm"] == arm]
        if len(arm_data) > 0:
            row = arm_data.iloc[0]
            fig_arch.add_trace(go.Bar(
                x=[f"{arch}<br>({arm})"],
                y=[row["mean"]],
                error_y=dict(type='data', array=[row["std"]]),
                name=arm,
                marker_color="#1f77b4" if arm == "Control" else "#2ca02c",
                text=[f"μ={row['mean']:.4f}<br>n={int(row['count'])}"],
                textposition="outside",
                hovertemplate="%{x}<br>Mean: %{y:.6f}<br>Std: %{error_y.array:.6f}<extra></extra>"
            ))

fig_arch.update_layout(
    title="Impact by Customer Archetype: Control vs Treatment",
    yaxis_title="Average Recommended Margin",
    xaxis_title="Archetype",
    barmode="group",
    template="plotly_white",
    height=500,
    width=1000,
    hovermode="x unified"
)

fig_arch.write_html(REPORTS_DIR / "policy_eval_archetype_impact.html")
print("✓ Saved: policy_eval_archetype_impact.html")

print("\n✓ All visualizations saved to reports/figures/")


✓ Saved: policy_eval_lift_bar.html
✓ Saved: policy_eval_cumulative_gain.html
✓ Saved: policy_eval_archetype_impact.html

✓ All visualizations saved to reports/figures/


## Section 10: Final Interpretation and Caveats

Document findings and explicit limitations for honest, publishable results.


In [57]:
# Section 10: Interpretation, Caveats, and Final Outputs

print("="*80)
print("MODULE 9: FINAL RESULTS SUMMARY")
print("="*80)

# Statistical test results
print("\n1. STATISTICAL TEST RESULTS (Welch's t-test)")
print("-" * 80)
print(f"   Sample size per group: {test_results['control_n']} (Control), {test_results['treatment_n']} (Treatment)")
print(f"   Control mean (Popularity): {test_results['control_mean']:.6f}")
print(f"   Treatment mean (Chimera): {test_results['treatment_mean']:.6f}")
print(f"   Point estimate lift: {test_results['lift']:.6f}")
print(f"   95% Confidence Interval: [{test_results['ci_lower']:.6f}, {test_results['ci_upper']:.6f}]")
print(f"   Two-sided p-value: {test_results['p_value']:.6f}")
print(f"   Effect size (Cohen's d): {test_results['cohens_d']:.4f}")
print(f"   Statistically significant? {'✓ YES (p < 0.05)' if test_results['is_significant'] else '✗ NO (p ≥ 0.05)'}")

# Targeting efficiency
print("\n2. BUDGET ALLOCATION (Top 20% Targeting)")
print("-" * 80)
print(f"   Households to target: {n_target}")
print(f"   Total margin delta from targeting: {gain_targeted:.6f}")
print(f"   Efficiency vs random: {gain_targeted/gain_random:.2f}x")
print(f"   This means: targeting the 'hottest' households yields {gain_targeted/gain_random:.2f}x more value")

# Key caveats
print("\n" + "="*80)
print("⚠️  IMPORTANT CAVEATS & INTERPRETATION GUIDE")
print("="*80)

print("\n1. THIS TEST EVALUATES RECOMMENDATION COMPOSITION, NOT ACTUAL PURCHASES")
print("   ✓ What we measured: The average normalized margin of the 5 items recommended")
print("   ✗ What we did NOT measure: Whether customers actually buy them, or uplift in revenue")
print("   → Implications: Results provide an upper-bound estimate of business impact")

print("\n2. ASSUMPTION: 100% ACCEPTANCE OF RECOMMENDATIONS")
print("   ✓ This is the best-case scenario")
print("   → Real conversion rate will scale results downward")
print("   → Example: If 10% click-through + 5% conversion, actual impact ≈ 0.05 × this value")

print("\n3. NO ARTIFICIAL SIMULATION NOISE")
print("   ✓ Outcomes derived directly from real recommendation rankings")
print("   ✓ No purchase probability modeling or random number generation")
print("   → Results are deterministic and reproducible")

print("\n4. POWER ANALYSIS INTERPRETATION")
print(f"   ✓ Current sample size: {n_per_group} per group")
print(f"   ✓ Minimum detectable effect: {mde:.6f}")
if abs(test_results['lift']) >= mde:
    print(f"   ✓ Observed effect IS large enough to detect at 80% power")
    print(f"     → Finding is reliable with current sample")
else:
    print(f"   ✗ Observed effect is NOT large enough to detect at 80% power")
    print(f"     → Would need {int(np.ceil((stats.norm.ppf(0.975) + stats.norm.ppf(0.80))**2 * 2 * std_pooled**2 / max(test_results['lift']**2, 1e-9)))} per group")
    print(f"     → Current sample provides directional signal only")

print("\n5. NEXT STEPS FOR VALIDATION")
print("   Step 1: Design live A/B test measuring actual purchase margins")
print("   Step 2: Implement Chimera for treatment cohort, Popularity for control")
print("   Step 3: Run test for sufficient duration (power analysis with realistic conversion)")
print("   Step 4: Compare results to this composition-based estimate")

print("\n" + "="*80)
print("BUSINESS RECOMMENDATION")
print("="*80)
if test_results["is_significant"] and test_results["lift"] > 0:
    print(f"\n✓ SIGNAL IS POSITIVE AND SIGNIFICANT")
    print(f"  Recommended action: Pilot Chimera with top 20% households (n={n_target})")
    print(f"  Expected structural benefit (upper bound): {test_results['lift']:.6f} margin index points")
    print(f"  Targeting efficiency: {gain_targeted/gain_random:.2f}x vs. random")
    print(f"  → Proceed to live test to measure actual purchase impact")
elif test_results["is_significant"] and test_results["lift"] < 0:
    print(f"\n✗ SIGNAL IS NEGATIVE")
    print(f"  Chimera is underperforming Popularity on margin composition")
    print(f"  Recommendation: Review utility function or model assumptions")
else:
    print(f"\n⚠️  SIGNAL IS INCONCLUSIVE")
    print(f"  Point estimate: {test_results['lift']:.6f} (could be 0)")
    print(f"  Not enough power with current sample to declare winner")
    print(f"  Recommendation: Collect more data or run live test")

# Save summary statistics
summary_output = pd.DataFrame({
    "Metric": [
        "Control Mean (Popularity)",
        "Treatment Mean (Chimera)",
        "Lift (Point Estimate)",
        "95% CI Lower",
        "95% CI Upper",
        "p-value (Two-Sided)",
        "Cohen's d",
        "Control Sample Size",
        "Treatment Sample Size",
        "Statistically Significant",
        "Minimum Detectable Effect (80% power)",
        "Top 20% Targeting Efficiency vs Random"
    ],
    "Value": [
        f"{test_results['control_mean']:.6f}",
        f"{test_results['treatment_mean']:.6f}",
        f"{test_results['lift']:.6f}",
        f"{test_results['ci_lower']:.6f}",
        f"{test_results['ci_upper']:.6f}",
        f"{test_results['p_value']:.6f}",
        f"{test_results['cohens_d']:.4f}",
        f"{test_results['control_n']}",
        f"{test_results['treatment_n']}",
        "Yes" if test_results["is_significant"] else "No",
        f"{mde:.6f}",
        f"{gain_targeted/gain_random:.2f}x"
    ]
})

summary_output.to_csv(DATA_DIR / "module9_ab_test_results.csv", index=False)
print(f"\n✓ Summary saved to: module9_ab_test_results.csv")
print(f"✓ Budget allocation saved to: module9_optimal_targeting_top20pct.csv")
print(f"✓ All visualizations saved to: {REPORTS_DIR}")


MODULE 9: FINAL RESULTS SUMMARY

1. STATISTICAL TEST RESULTS (Welch's t-test)
--------------------------------------------------------------------------------
   Sample size per group: 1204 (Control), 1204 (Treatment)
   Control mean (Popularity): 0.941362
   Treatment mean (Chimera): 0.988870
   Point estimate lift: 0.047508
   95% Confidence Interval: [0.041528, 0.053488]
   Two-sided p-value: 0.000000
   Effect size (Cohen's d): 0.6390
   Statistically significant? ✓ YES (p < 0.05)

2. BUDGET ALLOCATION (Top 20% Targeting)
--------------------------------------------------------------------------------
   Households to target: 481
   Total margin delta from targeting: 96.200000
   Efficiency vs random: 4.86x
   This means: targeting the 'hottest' households yields 4.86x more value

⚠️  IMPORTANT CAVEATS & INTERPRETATION GUIDE

1. THIS TEST EVALUATES RECOMMENDATION COMPOSITION, NOT ACTUAL PURCHASES
   ✓ What we measured: The average normalized margin of the 5 items recommended
   ✗ W